In [2]:
from pathlib import Path
import numpy as np
import torch

from data_utils import seed_everything, pairout_split, build_datasets_pairout
from evaluation import evaluate_model
from Models.AF_mamba import AFMamba

DATA_PATH = Path("Data/structured_dataset_1hz.pt")
CHECKPOINT_ROOT = Path("Trained_Models/af_mamba/paired_holdout")
BATCH_SIZE = 16
INPUT_SIZE = 3600
PREDICTION_HORIZON = 3600
SEED = 42

DATASET_NAMES = {
    "IRIDIA_new": "iridia_af",
    "LTAF_new": "ltaf",
    "MITAF_new": "mitbih_af",
    "NSR_new": "mitbih_nsr",
    "NSRRR_new": "nsr_rr",
}

PAIRED_HOLDOUTS = [
    ("iridia_af", "mitbih_nsr"),
    ("iridia_af", "nsr_rr"),
    ("ltaf", "mitbih_nsr"),
    ("ltaf", "nsr_rr"),
    ("mitbih_af", "mitbih_nsr"),
    ("mitbih_af", "nsr_rr"),
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed_everything(SEED)

data = torch.load(DATA_PATH, weights_only=False)

for sid in data:
    data[sid]["dataset"] = DATASET_NAMES.get(
        data[sid]["dataset"],
        data[sid]["dataset"]
    )

results = []

for heldout_af, heldout_nsr in PAIRED_HOLDOUTS:
    print(f"\n===== {heldout_af} + {heldout_nsr} =====")

    train_ids, val_ids, test_ids = pairout_split(data, heldout_af, heldout_nsr,
        input_segment_size=INPUT_SIZE, prediction_horizon=PREDICTION_HORIZON, seed=SEED)

    _, val_loader, test_loader, _, _ = build_datasets_pairout(data, train_ids, val_ids, test_ids,
        input_segment_size=INPUT_SIZE, prediction_horizon=PREDICTION_HORIZON, batch_size=BATCH_SIZE)

    model = AFMamba().to(device)
    checkpoint = CHECKPOINT_ROOT / f"{heldout_af}__{heldout_nsr}.pt"

    model.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True))

    val_metrics, *_ = evaluate_model(model, val_loader, device=device, threshold=None)
    threshold = val_metrics["threshold"]

    test_metrics, *_ = evaluate_model(model, test_loader, device=device, threshold=threshold)

    print(
        f"Sens={test_metrics['recall']:.4f}, "
        f"Spec={test_metrics['specificity']:.4f}, "
        f"F1={test_metrics['f1']:.4f}, "
        f"AUROC={test_metrics['roc_auc']:.4f}, "
        f"AUPRC={test_metrics['auprc']:.4f}"
    )

    results.append(test_metrics)

METRICS = {
    "Sensitivity": "recall",
    "Specificity": "specificity",
    "F1": "f1",
    "AUROC": "roc_auc",
    "AUPRC": "auprc",
}

print("\n===== PAIRED HOLDOUT RESULTS =====")
for label, key in METRICS.items():
    values = np.array([r[key] for r in results], dtype=float)
    print(f"{label}: {values.mean():.4f} ± {values.std():.4f}")

/usr/local/lib/python3.12/dist-packages/mamba_ssm/ops/selective_scan_interface.py:163: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd
/usr/local/lib/python3.12/dist-packages/mamba_ssm/ops/selective_scan_interface.py:239: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd
/usr/local/lib/python3.12/dist-packages/mamba_ssm/ops/triton/layer_norm.py:985: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd
/usr/local/lib/python3.12/dist-packages/mamba_ssm/ops/triton/layer_norm.py:1044: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd
/usr/local/lib/python3.12/dist-packages/mamba_ssm/dis


===== iridia_af + mitbih_nsr =====
Sens=0.8158, Spec=0.9972, F1=0.8964, AUROC=0.9912, AUPRC=0.9909

===== iridia_af + nsr_rr =====
Sens=0.5000, Spec=0.8937, F1=0.5124, AUROC=0.8363, AUPRC=0.5324

===== ltaf + mitbih_nsr =====
Sens=0.2500, Spec=1.0000, F1=0.4000, AUROC=0.9101, AUPRC=0.7272

===== ltaf + nsr_rr =====
Sens=0.7500, Spec=0.6811, F1=0.0545, AUROC=0.7971, AUPRC=0.0789

===== mitbih_af + mitbih_nsr =====
Sens=0.8750, Spec=1.0000, F1=0.9333, AUROC=0.9757, AUPRC=0.9401

===== mitbih_af + nsr_rr =====
Sens=0.9375, Spec=0.6058, F1=0.0726, AUROC=0.8729, AUPRC=0.1205

===== PAIRED HOLDOUT RESULTS =====
Sensitivity: 0.6880 ± 0.2396
Specificity: 0.8630 ± 0.1611
F1: 0.4782 ± 0.3496
AUROC: 0.8972 ± 0.0701
AUPRC: 0.5650 ± 0.3613
